In [86]:
import duckdb

In [87]:
DB_PATH = 'dados_duckdb.db'
con = duckdb.connect(database=DB_PATH, read_only=False)

In [79]:
df = con.execute('SELECT * FROM bronze_z0019').fetchdf()
df.head(6)

,NATBR,MARTX,WERKS,MAINS,LABST,nome_arquivo,nome_ingestão
0,10001,PARAFUSO,BT10,100,100,z0019_1.csv,2026-10-18 23:02:12.960986
1,10002,MARTELO,BT50,100,1500,z0019_1.csv,2026-10-18 23:02:12.960986
2,10003,PREGO,BT10,100,1500,z0019_1.csv,2026-10-18 23:02:12.960986
3,10004,SERRA,BT50,100,200,z0019_2.csv,2026-10-18 23:20:30.218906
4,10005,MACHADO,BR50,100,100,z0019_2.csv,2026-10-18 23:20:30.218906
5,10003,PREGO,BT10,100,60,z0019_2.csv,2026-10-18 23:20:30.218906


In [72]:
df = con.execute("""
SELECT *  FROM (
SELECT *, Row_Number() OVER (PARTITION BY NATBR ORDER BY nome_ingestão DESC) AS row
FROM bronze_z0019 where nome_ingestão >= '2026-10-18' 
) where row = 1
""").fetchdf()
df.head(6)

,NATBR,MARTX,WERKS,MAINS,LABST,nome_arquivo,nome_ingestão,row
0,10001,PARAFUSO,BT10,100,100,z0019_1.csv,2026-10-18 23:02:12.960986,1
1,10002,MARTELO,BT50,100,1500,z0019_1.csv,2026-10-18 23:02:12.960986,1
2,10004,SERRA,BT50,100,200,z0019_2.csv,2026-10-18 23:20:30.218906,1
3,10005,MACHADO,BR50,100,100,z0019_2.csv,2026-10-18 23:20:30.218906,1
4,10003,PREGO,BT10,100,60,z0019_2.csv,2026-10-18 23:20:30.218906,1


In [47]:
df_final = df.drop(columns=['nome_arquivo','nome_ingestão','row'])
df_final = df_final.rename(columns={'NATBR': 'id'})
df_final = df_final.rename(columns={'MARTX':'nm_produto'})
df_final = df_final.rename(columns={'WERKS':'id_categoria'})  
df_final = df_final.rename(columns={'MAINS':'id_fornecedor'})
df_final = df_final.rename(columns={'LABST':'vl_preco'})
df_final.head(6)

,id,nm_produto,id_categoria,id_fornecedor,vl_preco
0,10001,PARAFUSO,BT10,100,100
1,10002,MARTELO,BT50,100,1500
2,10004,SERRA,BT50,100,200
3,10005,MACHADO,BR50,100,100
4,10003,PREGO,BT10,100,60


In [57]:
df2 = df_final
df2 = df2.astype({'id': 'int64', 'nm_produto': 'string', 'id_categoria': 'string', 'id_fornecedor': 'string', 'vl_preco': 'float64'})
df2.dtypes
#df2.head(6)

id                 int64
nm_produto        string
id_categoria      string
id_fornecedor     string
vl_preco         float64
dtype: object

In [54]:
df_final.dtypes


id               str
nm_produto       str
id_categoria     str
id_fornecedor    str
vl_preco         str
dtype: object

In [88]:
con.execute ("""
CREATE TABLE IF NOT EXISTS produtos(
    id BIGINT, 
    nm_produto TEXT, 
    id_categoria TEXT, 
    id_fornecedor BIGINT, 
    vl_preco FLOAT
)""")

In [89]:
df2.head(6)

,id,nm_produto,id_categoria,id_fornecedor,vl_preco
0,10001,PARAFUSO,BT10,100,100.0
1,10002,MARTELO,BT50,100,1500.0
2,10004,SERRA,BT50,100,200.0
3,10005,MACHADO,BR50,100,100.0
4,10003,PREGO,BT10,100,60.0


In [ ]:
#SELECIONADA TUDO DE df2 E INSERIDA NA TABELA produtos
con.execute("INSERT INTO produtos SELECT * FROM df2")

In [92]:
df_resultado = con.execute("SELECT * FROM produtos").fetchdf()
df_resultado.head(10)

,id,nm_produto,id_categoria,id_fornecedor,vl_preco
0,10001,PARAFUSO,BT10,100,100.0
1,10002,MARTELO,BT50,100,1500.0
2,10004,SERRA,BT50,100,200.0
3,10005,MACHADO,BR50,100,100.0
4,10003,PREGO,BT10,100,60.0


In [93]:
con.close()